Daily **DAU**, **WAU**, and **MAU** from the semantic mart.

**Database connection** matches [`postgres-mcp-server/main.py`](../postgres-mcp-server/postgres-mcp-server/main.py): the first cell searches upward from the notebook working directory for **`postgres-mcp-server/.env`** (so it works when Jupyter’s cwd is the repo root or `notebooks/`). Fallback: a `.env` in the current directory. Required variables: **`DB_HOST`**, **`DB_PORT`**, **`DB_USER`**, **`DB_NAME`**. The password is read from the OS keyring — service **`futureproof_ds_db`**, username = **`DB_USER`** — same as the MCP server. Do not commit secrets.

In [ ]:
import os
from pathlib import Path
from typing import TypedDict

import keyring
import pandas as pd
import psycopg
import plotly.graph_objects as go
from dotenv import load_dotenv
from IPython.display import display
from plotly.subplots import make_subplots


# Find postgres-mcp-server/.env whether cwd is repo root, notebooks/, etc.
def _load_dotenv_for_notebook() -> None:
    here = Path.cwd().resolve()
    for root in [here, *here.parents]:
        candidate = root / "postgres-mcp-server" / ".env"
        if candidate.is_file():
            load_dotenv(candidate)
            return
    load_dotenv()


_load_dotenv_for_notebook()

In [ ]:
QUERY = """
SELECT date, dau, wau, mau
FROM marts.user_activity_metrics
ORDER BY date
"""


def _require_env(name: str) -> str:
    value = os.getenv(name)
    if value is None or value == "":
        raise RuntimeError(f"Missing required environment variable: {name}")
    return value


class DBConfig(TypedDict):
    host: str
    port: int
    user: str
    password: str
    dbname: str


def _db_config() -> DBConfig:
    """Same as postgres-mcp-server/main.py."""
    db_user = _require_env("DB_USER")
    password = keyring.get_password("futureproof_ds_db", db_user)
    if password is None or password == "":
        raise RuntimeError(
            "Database password not found in keyring "
            "(service 'futureproof_ds_db', username = DB_USER)."
        )
    return {
        "host": _require_env("DB_HOST"),
        "port": int(_require_env("DB_PORT")),
        "user": db_user,
        "password": password,
        "dbname": _require_env("DB_NAME"),
    }


def connect() -> psycopg.Connection:
    """Match MCP server's psycopg.connect(host=..., port=..., ...)."""
    cfg = _db_config()
    return psycopg.connect(
        host=cfg["host"],
        port=cfg["port"],
        user=cfg["user"],
        password=cfg["password"],
        dbname=cfg["dbname"],
    )


with connect() as conn:
    df = pd.read_sql_query(QUERY, conn)

df["date"] = pd.to_datetime(df["date"])
df = df.set_index("date").sort_index()
df.head()

In [ ]:
display(df[["dau", "wau", "mau"]].describe())

In [ ]:
fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    subplot_titles=("DAU", "WAU", "MAU"),
)
colors = {"dau": "#2563eb", "wau": "#059669", "mau": "#d97706"}
for row, col in enumerate(["dau", "wau", "mau"], start=1):
    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=df[col],
            mode="lines",
            name=col.upper(),
            line=dict(color=colors[col], width=2),
            showlegend=False,
        ),
        row=row,
        col=1,
    )
fig.update_layout(
    height=720,
    title_text="User activity metrics over time",
    hovermode="x unified",
    template="plotly_white",
)
fig.show()

In [ ]:
roll = df["dau"].rolling(7, min_periods=1).mean()
fig2 = go.Figure()
fig2.add_trace(
    go.Scatter(
        x=df.index, y=df["dau"], mode="lines", name="DAU", line=dict(color="#2563eb")
    )
)
fig2.add_trace(
    go.Scatter(
        x=df.index,
        y=roll,
        mode="lines",
        name="DAU (7-day rolling mean)",
        line=dict(color="#93c5fd", width=2),
    )
)
fig2.update_layout(
    title="DAU with 7-day rolling average",
    hovermode="x unified",
    template="plotly_white",
    yaxis_title="Users",
    xaxis_title="Date",
)
fig2.show()